Optimize the k-value for k-NN using the Elbow Method by manually plotting error rates for different k-values, ensuring a statistically sound selection without using built-in visualization libraries.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].dropna()

X = df[features].astype(float).to_numpy()
y = df[target].astype(int).to_numpy()

print("Dataset shape:", df.shape)
print("Total records:", len(df))

Dataset shape: (79, 6)
Total records: 79


In [2]:
np.random.seed(42)

class_indices = {}

for label in np.unique(y):
    indices = np.where(y == label)[0].copy()
    np.random.shuffle(indices)
    class_indices[label] = np.array_split(indices, 5)

folds = []

for i in range(5):
    fold = np.concatenate([
        class_indices[label][i]
        for label in class_indices
    ])
    np.random.shuffle(fold)
    folds.append(fold)

print("Created 5 stratified folds.")

Created 5 stratified folds.


In [3]:
def knn_predict(X_train, y_train, X_test, k):
    predictions = []

    train_squared = np.sum(X_train ** 2, axis=1)

    for row in X_test:
        distances = (
            np.sum(row ** 2)
            + train_squared
            - 2 * (X_train @ row)
        )

        distances = np.maximum(distances, 0)
        nearest = np.argpartition(distances, k - 1)[:k]
        nearest = nearest[np.argsort(distances[nearest])]

        labels = y_train[nearest]
        classes, counts = np.unique(labels, return_counts=True)
        tied = classes[counts == counts.max()]

        if len(tied) == 1:
            prediction = tied[0]
        else:
            totals = [
                np.sum(distances[nearest][labels == label])
                for label in tied
            ]
            prediction = tied[np.argmin(totals)]

        predictions.append(prediction)

    return np.array(predictions)

In [4]:
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
fold_errors = {k: [] for k in k_values}

for fold_number in range(5):
    test_indices = folds[fold_number]
    train_indices = np.concatenate([
        folds[i] for i in range(5) if i != fold_number
    ])

    X_train = X[train_indices].copy()
    X_test = X[test_indices].copy()
    y_train = y[train_indices]
    y_test = y[test_indices]

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    for k in k_values:
        if k > len(X_train):
            continue

        predictions = knn_predict(X_train, y_train, X_test, k)
        error_rate = np.mean(predictions != y_test) * 100
        fold_errors[k].append(error_rate)

    print("Completed fold:", fold_number + 1)

results = []

for k in k_values:
    errors = fold_errors[k]

    results.append({
        "K": k,
        "Mean Error (%)": np.mean(errors),
        "Std Error (%)": np.std(errors, ddof=1)
    })

results_df = pd.DataFrame(results)

print("\nCross-Validation Error Rates")
print(results_df.round(2).to_string(index=False))

Completed fold: 1
Completed fold: 2
Completed fold: 3
Completed fold: 4
Completed fold: 5

Cross-Validation Error Rates
 K  Mean Error (%)  Std Error (%)
 1           21.76          11.81
 3           24.20           6.03
 5           26.71           3.87
 7           25.37           1.33
 9           22.71           2.54
11           22.71           2.54
13           22.71           2.54
15           22.71           2.54


In [5]:
valid_results = results_df.dropna().reset_index(drop=True)

errors = valid_results["Mean Error (%)"].to_numpy()
ks = valid_results["K"].to_numpy()

max_error = max(errors) + 1
min_error = max(0, min(errors) - 1)

height = 10
levels = np.linspace(max_error, min_error, height)

print("\nElbow Method: K vs Mean Error Rate\n")

for level in levels:
    line = f"{level:6.1f} |"

    for error in errors:
        if abs(error - level) <= (max_error - min_error) / height / 2:
            line += "  *  "
        else:
            line += "     "

    print(line)

print("       +" + "-----" * len(ks))
print("        " + "".join(f"{k:^5}" for k in ks))
print("                 K value")


Elbow Method: K vs Mean Error Rate

  27.7 |                                        
  26.9 |            *                           
  26.2 |                                        
  25.4 |                 *                      
  24.6 |                                        
  23.8 |                                        
  23.1 |                                        
  22.3 |                                        
  21.5 |  *                                     
  20.8 |                                        
       +----------------------------------------
          1    3    5    7    9   11   13   15  
                 K value


In [6]:
results_df["Standard Error (%)"] = (
    results_df["Std Error (%)"] / np.sqrt(5)
)

best_row = results_df.loc[results_df["Mean Error (%)"].idxmin()]
threshold = (
    best_row["Mean Error (%)"] + best_row["Standard Error (%)"]
)

eligible = results_df[
    results_df["Mean Error (%)"] <= threshold
]

selected_k = int(eligible["K"].max())

print("Minimum-error K:", int(best_row["K"]))
print("Minimum mean error:", round(best_row["Mean Error (%)"], 2), "%")
print("One-standard-error threshold:", round(threshold, 2), "%")
print("Selected K:", selected_k)

Minimum-error K: 1
Minimum mean error: 21.76 %
One-standard-error threshold: 27.04 %
Selected K: 15


In [7]:
print("Its mean cross-validation error:",
      round(results_df.loc[results_df["K"] == selected_k,
                           "Mean Error (%)"].iloc[0], 2), "%")

Its mean cross-validation error: 22.71 %
